# Q Learning with Frozen Lake

In this Notebook, we'll implement a Q Learning agent <b>that learns to play FrozenLake.</b>

The goal of this game is <b>to move the character from the starting state (S) to the goal state (G)</b> by walking only on frozen tiles (F) and avoiding holes (H). 

Frozen Lake has two modes that you can play in: 
1. Deterministic environment -- wherever the agent chooses to move is how it moves.
2. Stochastic environment -- the ice is slippery, **so you can slide and won't always move in the direction you intend.** In the stochastic environment, you have a 1/3 chance of moving in the intended direction, and a 1/3 chance in moving in either of the perpedicular directions to your intended motion.

We will experiment with both modes.

<img src="ims/frozen_lake.gif" alt="Environments"/>

### Import the dependencies

We use 3 libraries:

- Numpy for our Qtable
- OpenAI Gym for our FrozenLake Environment
- Random to generate random numbers
- matplotlib to visualise our environment

In [ ]:
!pip install gym

import numpy as np
import gym
import random
import matplotlib.pyplot as plt

## Step 1: Create the environment 🎮
- Here we'll create the FrozenLake 4x4 environment. 
- OpenAI Gym is a library <b> composed of many environments that we can use to train our agents.</b>
- In our case we choose to use Frozen Lake.

### Action Space
The agent takes a 1-element vector for actions. The action space is (direction), where it decides direction to move in which can be:

- 0: LEFT
- 1: DOWN
- 2: RIGHT
- 3: UP


### Observation Space
The observation is a value representing the agent’s current position as current_row * nrows + current_col (where both the row and col start at 0). For example, the goal position in the 4x4 map can be calculated as follows: 3 * 4 + 3 = 15. The number of possible observations is dependent on the size of the map. For example, the 4x4 map has 16 possible observations. For example this is what state 0 looks like:

<img src="ims/frozenlake.png" alt="FrozenLake">

### Rewards
Reward schedule:
- Reach goal (G): +1
- Reach hole (H): 0
- Reach frozen (F): 0

In [ ]:
env = gym.make("FrozenLake-v1", render_mode = 'rgb_array', map_name = "4x4", is_slippery = False)

env.reset()
plt.imshow(env.render())
plt.show()

## Step 2: Create the Q-table and initialize it 🗄️
Now, we'll create our Q-table. To know how many rows (states) and columns (actions) we need, we need to calculate the action_size and the state_size. OpenAI Gym provides us a way to do that: `env.action_space.n` and `env.observation_space.n`

Use this information to initialise the correct size grid with zeros for the Q-table. 
**Hint: Use np.zeros().** 

Remember in the beginning, our Q-Table is useless since it gives arbitrary value for each state-action pair (most of the time we initialize the Q-Table to 0 values). But, as we’ll explore the environment and update our Q-Table it will give us better and better approximations.

Note how the Q-Table is structured in the example below and create a similar table for your FrozenLake environment below.

<img src="ims/qlearning.jpeg" alt="q-learning.jpeg" width="100%"/>


In [ ]:
action_size = env.action_space.n
state_size = env.observation_space.n

# Create our Q table with state_size rows and action_size columns
qtable = # TODO: Initialise the Q-Table with zeros
print(qtable)

## Lets try and play the game where we select actions randomly and see what our success rate is of winning the game.

In [ ]:
#reset the environment back to the beginning state
env.reset()

successes = 0
num_test_episodes = 10
for episode in range(num_test_episodes):
    state = env.reset()[0]
    step = 0
    done = False
    print("****************************************************")
    print("EPISODE ", episode)

    while not done:
        
        # Sample a random action to take in the environment
        action = env.action_space.sample()
        
        new_state, reward, done, _, info = env.step(action)
        step += 1
        if done:
            # Here, we decide to only print the last state (to see if our agent is on the goal or fall into an hole)
            if new_state == 15:
                print("We reached our Goal 🏆")
                successes += 1
            else:
                print("We fell into a hole ☠️")
            
            # We print the number of step it took.
            print("Number of steps", step)
            
            break
        state = new_state
        
print("Success Rate: ", (successes/num_test_episodes)*100 , "%")
env.close()

A random agent doesn't work very well! Let's instead try to train an agent with the Q learning algorithm!

## Step 3: The Q learning algorithm 🧠 

### Create the hyperparameters ⚙️
Here, we'll specify the hyperparameters that control Q-learning.

In [ ]:
total_episodes = 1000       # Total episodes to train for
learning_rate = 0.7          # Learning rate
max_steps = 99              # Max steps per episode
gamma = 0.95                 # Discounting rate

# Exploration parameters
epsilon = 1.0                 # Exploration rate
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.01            # Minimum exploration probability 
decay_rate = 0.005             # Exponential decay rate for exploration prob

### Implementing the Q learning algorithm!
Now we can implement the Q learning algorithm shown below:

1. Initialize Q-values (Q(s,a)) for all state-action pairs.
2. For a certain number of episodes until learning is stopped...
    1. Reset the environment.
    2. While the episode isn't over or max_steps is not exceeded: 
        1. Choose an action (a) in the current world state (s) -- exploration vs exploitation
        3. Take the action (a) and observe the outcome state (s') and reward (r)
        4. Update the Q-values (Q(s,a)) using the Bellman equation below.
        5. Update the state to the outcome state.
    3. Reduce the exploration rate (epsilon) - this has been implemented for you.

![alt text](ims/Bellman.jpg)
  
Adapt the code below to implement the Q learning algorithm. 

In [ ]:
rewards = []

for episode in range(total_episodes):
    # Reset the environment
    state = env.reset()[0]
    step = 0
    done = False
    total_rewards = 0
    
    for step in range(max_steps):
        # Choose an action a in the current world state (s)
        ## First we randomize a number
        exp_exp_tradeoff = random.uniform(0, 1)
        
        # If this number > greater than epsilon --> exploitation (taking the biggest Q value for this state)
        if exp_exp_tradeoff > epsilon:
            action = #TO DO: Choose exploitation

        # Else doing a random choice --> exploration
        else:
            action = #TO DO: Choose exploration
        
        # TO DO: add q-learning code here
        
    # Reduce epsilon (because we need less and less exploration)
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode) 
    rewards.append(total_rewards)
    

## Step 4: Use our Q-table to play FrozenLake ! 👾
- After all the episodes, our Q-table can be used as a "cheatsheet" to play FrozenLake
- By running this cell you can see the result of our agent playing FrozenLake.
- Note how the success rate is now much higher that when we acted randomly.
- If your agent has successfully learned the Q-Values you should note an increase in the success rate.
- Experiment with the hyperparameters to understand how they impact performance.

In [ ]:
env.reset()

successes = 0
num_test_episodes = 10
for episode in range(num_test_episodes):
    state = env.reset()[0]
    step = 0
    done = False
    print("****************************************************")
    print("EPISODE ", episode)

    for step in range(max_steps):
        
        # Take the action (index) that have the maximum expected future reward given that state
        action = np.argmax(qtable[state,:])
        
        new_state, reward, done, _, info = env.step(action)
        
        if done:
            # Here, we decide to only print the last state (to see if our agent is on the goal or fall into an hole)
            # env.render()
            if new_state == 15:
                print("We reached our Goal 🏆")
                successes += 1
            else:
                print("We fell into a hole ☠️")
            
            # We print the number of step it took.
            print("Number of steps", step)
            
            break
        state = new_state
        
print("Success Rate: ", (successes/num_test_episodes)*100 , "%")
env.close()

## Step 6: A stochastic environment!

Notice how the agent takes the same actions and always reaches the goal? This is because it is a deterministic environment! 

Go back and re-initialise the environment with is_slippery = True! This will create a stochastic environment.

**Consider:**
- How does the performance of your RL agent, using the same hyperparameters during learning, change? Why is this?

## Step 7: A larger environment!

Below, we're creating a larger environment! Re-run the code to find a Q learning agent for this new environment.

**Consider:**
- How does the performance of your RL agent, using the same hyperparameters, change? Why is this?

With this in mind, what are some limitations of Q learning? And Reinforcement Learning in general?

In [ ]:
new_map = [
    "SFFFFH",
    "FFFFFF",
    "FFHFFF",
    "FHFFFF",
    "FHFFFF",
    "FFFFFG",
]

env = gym.make("FrozenLake-v1", render_mode = 'rgb_array', desc = new_map, is_slippery = True)

env.reset()
plt.imshow(env.render())
plt.show()